In [ ]:
import pandas as pd
import numpy as np
import re

df = pd.read_csv(
    "/content/IMDB Dataset.csv",
    engine="python"
)

print("Dataset loaded successfully!")
print("Shape:", df.shape)

display(df.head())

Dataset loaded successfully!
Shape: (50000, 2)


,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [ ]:
print("Dataset shape:", df.shape)

print("\nColumns:")
print(df.columns.tolist())

print("\nMissing values:")
print(df.isnull().sum())

print("\nSentiment distribution:")
print(df["sentiment"].value_counts())

Dataset shape: (50000, 2)

Columns:
['review', 'sentiment']

Missing values:
review       0
sentiment    0
dtype: int64

Sentiment distribution:
sentiment
positive    25000
negative    25000
Name: count, dtype: int64


In [ ]:
# Count exact duplicate rows
print("Exact duplicate rows:", df.duplicated().sum())

# Count reviews that appear more than once
print("Duplicate review texts:", df["review"].duplicated().sum())

# Check whether duplicated reviews have conflicting sentiment labels
duplicate_reviews = df[df["review"].duplicated(keep=False)]

label_conflicts = (
    duplicate_reviews
    .groupby("review")["sentiment"]
    .nunique()
)

print("Reviews with conflicting labels:", (label_conflicts > 1).sum())

Exact duplicate rows: 418
Duplicate review texts: 418
Reviews with conflicting labels: 0


In [ ]:
from bs4 import BeautifulSoup

In [ ]:
def clean_review(text):
    text = BeautifulSoup(text, "html.parser").get_text(" ")# 1. Remove HTML tags
    text = text.lower()# 2. Convert text to lowercase
    text = re.sub(r"http\S+|www\S+", " ", text)# 3. Remove URLs
    text = re.sub(r"[^a-z\s']", " ", text)# 4. Keep letters, spaces and apostrophes
    # 5. Remove extra whitespace
    text = re.sub(r"\s+", " ", text).strip()
    return text

In [ ]:
sample_review = """
This movie was NOT good!!! <br /><br />
I don't recommend it. Visit https://example.com
"""

cleaned_sample = clean_review(sample_review)

print("Original review:")
print(sample_review)

print("\nCleaned review:")
print(cleaned_sample)

Original review:

This movie was NOT good!!! <br /><br />
I don't recommend it. Visit https://example.com


Cleaned review:
this movie was not good i don't recommend it visit


In [ ]:
df["clean_review"] = df["review"].apply(clean_review)

print("Original review:")
print(df["review"].iloc[0])

print("\nCleaned review:")
print(df["clean_review"].iloc[0])

Original review:
One of the other reviewers has mentioned that after watching just 1 Oz episode you'll be hooked. They are right, as this is exactly what happened with me.<br /><br />The first thing that struck me about Oz was its brutality and unflinching scenes of violence, which set in right from the word GO. Trust me, this is not a show for the faint hearted or timid. This show pulls no punches with regards to drugs, sex or violence. Its is hardcore, in the classic use of the word.<br /><br />It is called OZ as that is the nickname given to the Oswald Maximum Security State Penitentary. It focuses mainly on Emerald City, an experimental section of the prison where all the cells have glass fronts and face inwards, so privacy is not high on the agenda. Em City is home to many..Aryans, Muslims, gangstas, Latinos, Christians, Italians, Irish and more....so scuffles, death stares, dodgy dealings and shady agreements are never far away.<br /><br />I would say the main appeal of the show 

In [ ]:
empty_after_cleaning = (df["clean_review"].str.strip() == "").sum()

print("Empty reviews after cleaning:", empty_after_cleaning)

Empty reviews after cleaning: 0


In [ ]:
from collections import Counter

example_sentences = [
    "this movie is good",
    "this movie is not good",
    "the movie is amazing"
]

# Break sentences into words
tokenized_sentences = [
    sentence.split() for sentence in example_sentences
]

# Count word frequencies
word_counts = Counter(
    word
    for sentence in tokenized_sentences
    for word in sentence
)

print("Tokenized sentences:")
print(tokenized_sentences)

print("\nWord frequencies:")
print(word_counts)

Tokenized sentences:
[['this', 'movie', 'is', 'good'], ['this', 'movie', 'is', 'not', 'good'], ['the', 'movie', 'is', 'amazing']]

Word frequencies:
Counter({'movie': 3, 'is': 3, 'this': 2, 'good': 2, 'not': 1, 'the': 1, 'amazing': 1})


In [ ]:
# Reserve special tokens
word_to_id = {
    "<PAD>": 0,
    "<UNK>": 1
}

# Assign a unique ID to each word
for word in word_counts:
    word_to_id[word] = len(word_to_id)

print(word_to_id)

{'<PAD>': 0, '<UNK>': 1, 'this': 2, 'movie': 3, 'is': 4, 'good': 5, 'not': 6, 'the': 7, 'amazing': 8}


In [ ]:
sentence = "this movie is not good"
encoded_sentence = [word_to_id.get(word,word_to_id['<UNK>']) for word in sentence.split()
]
print("Original:", sentence)
print("Encoded:", encoded_sentence)

Original: this movie is not good
Encoded: [2, 3, 4, 6, 5]


In [ ]:
from tensorflow.keras.preprocessing.sequence import pad_sequences

encoded_sentences = [
    [2, 3, 4, 5],
    [2, 3, 4, 6, 5],
    [7, 3, 4, 8]
]

padded = pad_sequences(
    encoded_sentences,
    maxlen=5,
    padding="post",
    value=0
)

print(padded)

[[2 3 4 5 0]
 [2 3 4 6 5]
 [7 3 4 8 0]]


In [ ]:
# Count tokens in each cleaned review
df["token_count"] = df["clean_review"].str.split().str.len()

print(df["token_count"].describe(percentiles=[0.50, 0.75, 0.90, 0.95, 0.99]))

# Check truncation coverage
for length in [200, 300, 400, 500]:
    truncated = (df["token_count"] > length).mean() * 100
    print(f"Max length {length}: {truncated:.2f}% reviews truncated")

count    50000.000000
mean       229.745920
std        170.179236
min          6.000000
50%        172.000000
75%        279.000000
90%        448.000000
95%        586.000000
99%        899.010000
max       2462.000000
Name: token_count, dtype: float64
Max length 200: 40.54% reviews truncated
Max length 300: 22.02% reviews truncated
Max length 400: 12.77% reviews truncated
Max length 500: 7.68% reviews truncated


In [ ]:
from sklearn.model_selection import train_test_split

In [ ]:
df["label"] = df["sentiment"].map({
    "negative": 0,
    "positive": 1
})
# Separate input text and target labels
X = df["clean_review"]
y = df["label"]
# 80% training, 20% temporary
X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)
# then divide temporary data equally
X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    random_state=42,
    stratify=y_temp
)
print("Training:", X_train.shape)
print("Validation:", X_val.shape)
print("Testing:", X_test.shape)

Training: (40000,)
Validation: (5000,)
Testing: (5000,)


In [ ]:
from tensorflow.keras.preprocessing.text import Tokenizer

MAX_WORDS = 30000

tokenizer = Tokenizer(
    num_words=MAX_WORDS,
    oov_token="<OOV>",
    filters=""
)

# Learn vocabulary ONLY from training reviews
tokenizer.fit_on_texts(X_train)

print("Total unique words:", len(tokenizer.word_index))

Total unique words: 107696


In [ ]:
print("Most frequent words:")

for word, index in list(tokenizer.word_index.items())[:20]:
    print(word, ":", index)

Most frequent words:
<OOV> : 1
the : 2
and : 3
a : 4
of : 5
to : 6
is : 7
in : 8
it : 9
i : 10
this : 11
that : 12
was : 13
as : 14
for : 15
with : 16
movie : 17
but : 18
film : 19
on : 20


In [ ]:
tokenizer.word_index.items()

dict_items([('<OOV>', 1), ('the', 2), ('and', 3), ('a', 4), ('of', 5), ('to', 6), ('is', 7), ('in', 8), ('it', 9), ('i', 10), ('this', 11), ('that', 12), ('was', 13), ('as', 14), ('for', 15), ('with', 16), ('movie', 17), ('but', 18), ('film', 19), ('on', 20), ('not', 21), ('you', 22), ('are', 23), ('his', 24), ('have', 25), ('be', 26), ('one', 27), ('he', 28), ('all', 29), ('at', 30), ('by', 31), ('an', 32), ('they', 33), ('so', 34), ('who', 35), ('from', 36), ('like', 37), ('or', 38), ('just', 39), ('her', 40), ('out', 41), ('if', 42), ('about', 43), ("it's", 44), ('has', 45), ('there', 46), ('some', 47), ('what', 48), ('good', 49), ('when', 50), ('more', 51), ('very', 52), ('up', 53), ('no', 54), ('time', 55), ('even', 56), ('my', 57), ('would', 58), ('she', 59), ('which', 60), ('really', 61), ('only', 62), ('see', 63), ('story', 64), ('their', 65), ('had', 66), ('can', 67), ('me', 68), ('were', 69), ('well', 70), ('we', 71), ('much', 72), ('than', 73), ('bad', 74), ('been', 75), ('g

In [ ]:
# Convert text into sequences of integer IDs

X_train_seq = tokenizer.texts_to_sequences(X_train)
X_val_seq = tokenizer.texts_to_sequences(X_val)
X_test_seq = tokenizer.texts_to_sequences(X_test)

print("First training review:")
print(X_train.iloc[0][:200])

print("\nEncoded sequence:")
print(X_train_seq[0][:30])

print("\nSequence length:", len(X_train_seq[0]))

First training review:
i caught this little gem totally by accident back in or ' i was at a revival theatre to see two old silly sci fi movies the theatre was packed full and with no warning they showed a bunch of sci fi sh

Encoded sequence:
[10, 1044, 11, 119, 1499, 454, 31, 1594, 142, 8, 38, 426, 10, 13, 30, 4, 8587, 1658, 6, 63, 105, 157, 666, 862, 860, 97, 2, 1658, 13, 2809]

Sequence length: 163


In [ ]:
from tensorflow.keras.preprocessing.sequence import pad_sequences

MAX_LEN = 300

X_train_pad = pad_sequences(
    X_train_seq,
    maxlen=MAX_LEN,
    padding="post",
    truncating="post"
)

X_val_pad = pad_sequences(
    X_val_seq,
    maxlen=MAX_LEN,
    padding="post",
    truncating="post"
)

X_test_pad = pad_sequences(
    X_test_seq,
    maxlen=MAX_LEN,
    padding="post",
    truncating="post"
)

print("Training shape:", X_train_pad.shape)
print("Validation shape:", X_val_pad.shape)
print("Testing shape:", X_test_pad.shape)

Training shape: (40000, 300)
Validation shape: (5000, 300)
Testing shape: (5000, 300)


In [ ]:
# Calculate actual token counts after truncation
actual_lengths = [min(len(seq), MAX_LEN) for seq in X_train_seq]

print("Average actual tokens:", np.mean(actual_lengths))
print("Average padding tokens:", MAX_LEN - np.mean(actual_lengths))

Average actual tokens: 188.619725
Average padding tokens: 111.38027500000001


In [ ]:
oov_id = tokenizer.word_index["<OOV>"]

oov_count = sum(seq.count(oov_id) for seq in X_train_seq)
total_tokens = sum(len(seq) for seq in X_train_seq)

print("OOV tokens:", oov_count)
print("Total tokens:", total_tokens)
print("OOV percentage:", round(oov_count / total_tokens * 100, 2), "%")

OOV tokens: 158301
Total tokens: 9198724
OOV percentage: 1.72 %


In [ ]:
print("Training labels:")
print(y_train.value_counts())

print("\nValidation labels:")
print(y_val.value_counts())

print("\nTesting labels:")
print(y_test.value_counts())

Training labels:
label
1    20000
0    20000
Name: count, dtype: int64

Validation labels:
label
0    2500
1    2500
Name: count, dtype: int64

Testing labels:
label
0    2500
1    2500
Name: count, dtype: int64


In [ ]:
# Find a short review
short_index = next(
    i for i, seq in enumerate(X_train_seq)
    if len(seq) < MAX_LEN
)

# Find a long review
long_index = next(
    i for i, seq in enumerate(X_train_seq)
    if len(seq) > MAX_LEN
)

print("Short review:")
print("Original length:", len(X_train_seq[short_index]))
print("Padded length:", len(X_train_pad[short_index]))
print("Padding count:", np.sum(X_train_pad[short_index] == 0))

print("\nLong review:")
print("Original length:", len(X_train_seq[long_index]))
print("After truncation:", len(X_train_pad[long_index]))

Short review:
Original length: 163
Padded length: 300
Padding count: 137

Long review:
Original length: 317
After truncation: 300


In [ ]:
import pickle
import os

os.makedirs("/content/SentimentForge/models", exist_ok=True)

tokenizer_path = "/content/SentimentForge/models/tokenizer.pkl"

with open(tokenizer_path, "wb") as file:
    pickle.dump(tokenizer, file)

print("Tokenizer saved successfully!")

Tokenizer saved successfully!


In [ ]:
import json

config = {
    "max_words": MAX_WORDS,
    "max_length": MAX_LEN,
    "padding": "post",
    "truncating": "post",
    "oov_token": "<OOV>",
    "negative_label": 0,
    "positive_label": 1
}

config_path = "/content/SentimentForge/models/preprocessing_config.json"

with open(config_path, "w") as file:
    json.dump(config, file, indent=4)

print("Configuration saved successfully!")

Configuration saved successfully!


In [ ]:
print(os.listdir("/content/SentimentForge/models"))

['tokenizer.pkl', 'preprocessing_config.json']
